# Report of Deforestation Analysis and LULC Benchmarking

This report summarises the findings of the first stage of the deforestation method review:

- **Part I – Hansen Global Forest Change** (`deforestation_hansen.ipynb`, `investigate_tier.ipynb`,
  `hansen_annual_maps.ipynb`): how much forest Guinea had and lost, how sensitive the numbers are to the forest
  definition and to Hansen's 2013 method change, and where the loss happens.
- **Part II – Land-cover benchmark** (`lulc_benchmark.ipynb`): how the sample of points works, and how much the
  land-cover and forest products disagree.

**Conventions**

- All areas are in **hectares (ha)**.
- Non-spatial plots are interactive (plotly); maps are static images (matplotlib).
- Everything is read from the tables already saved by those notebooks. The only new Earth Engine computation is the
  ~1 km² hexagon table of section 0.3, run once (~0.8 EECU-hours) and cached; re-running the report then costs no
  Earth Engine compute.

## Setup

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import ee
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from h3.api import basic_int as h3_int
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from plotly.subplots import make_subplots
from shapely.geometry import Polygon, box

PROJECT_ROOT = Path.cwd().parent                     # .../Guinea_Methodology
KEY_FILE = PROJECT_ROOT / "gee-account.json"         # service-account key (git-ignored), only needed for section 0.3
OUTPUT_DIR = Path.cwd() / "output_datasets"
BENCHMARK_DIR = OUTPUT_DIR / "lulc_benchmark"

ECOREGIONS_GPKG = OUTPUT_DIR / "ecoregions_guinea.gpkg"                            # ecoregions.ipynb
ECOREGION_POLYGONS_GPKG = OUTPUT_DIR / "ecoregion_polygons.gpkg"                   # ecoregions.ipynb
HANSEN_LOSS_PARQUET = OUTPUT_DIR / "hansen_loss_by_polygon_cell_year.parquet"      # deforestation_hansen.ipynb
HANSEN_RELEASES_PARQUET = OUTPUT_DIR / "hansen_loss_by_release_year.parquet"       # hansen_annual_maps.ipynb
SAMPLE_POINTS_PARQUET = BENCHMARK_DIR / "sample_points_h3res7.parquet"            # lulc_benchmark.ipynb

In [ ]:
HANSEN_ASSET = "UMD/hansen/global_forest_change_2025_v1_13"
CANOPY_THRESHOLDS = [0, 10, 30, 50]   # thresholds saved by deforestation_hansen.ipynb
CANOPY_THRESHOLD = 30                 # adopted forest definition for Hansen (section 2)
FIRST_YEAR, LAST_YEAR = 2001, 2025
METHOD_CHANGE_YEAR = 2013             # Hansen method change: analysis starts here
RECENT_YEAR = 2020                    # second focus period: 2020 onwards
COMPARISON_YEAR = 2020                # year of the land-cover benchmark

FINE_H3_RESOLUTION = 8                # chosen in section 0.2
GRID_DEG = 0.00125                    # Earth Engine summary grid: exactly 5 x 5 Hansen pixels (~1.9 ha)
MAX_EECU_HOURS = 2                    # cost limit for section 0.3
FORCE_RECOMPUTE = False               # True = run section 0.3 on Earth Engine again even if the parquet exists

H3_FINE_PARQUET = OUTPUT_DIR / f"hansen_forest_loss_h3res{FINE_H3_RESOLUTION}.parquet"
H3_FINE_RUN_JSON = OUTPUT_DIR / f"hansen_forest_loss_h3res{FINE_H3_RESOLUTION}.run.json"
GRID_TILES_DIR = OUTPUT_DIR / "hansen_grid_tiles"      # .npz cache of the Earth Engine tiles (git-ignored)

M2_PER_HA, HA_PER_KM2 = 10_000, 100
PLOT_TEMPLATE = "plotly_white"
PERIOD_COLORS = {f"{FIRST_YEAR}–{METHOD_CHANGE_YEAR - 1}": "#b0b0b0",
                 f"{METHOD_CHANGE_YEAR}–{RECENT_YEAR - 1}": "#e67e22",
                 f"{RECENT_YEAR}–{LAST_YEAR}": "#c0392b"}
THRESHOLD_COLORS = {0: "#c7e9c0", 10: "#74c476", 30: "#238b45", 50: "#00441b"}

### Common data

In [ ]:
guinea_boundary = gpd.read_file(ECOREGIONS_GPKG, layer="guinea_boundary")
ecoregions = gpd.read_file(ECOREGIONS_GPKG, layer="ecoregions_guinea")          # one row per ecoregion
polygons = gpd.read_file(ECOREGION_POLYGONS_GPKG).reset_index(names="polygon_index")  # one row per ecoregion part

ECOREGION_NAMES = sorted(ecoregions["ECO_NAME"])
ECOREGION_COLORS = dict(zip(ECOREGION_NAMES, px.colors.qualitative.Safe))  # as in the other notebooks
ECOREGION_COLORS_MPL = {name: tuple(v / 255 for v in px.colors.unlabel_rgb(color))  # same colours for matplotlib
                        for name, color in ECOREGION_COLORS.items()}
ECOREGION_SHORT = {
    "Guinean forest-savanna": "Forest-savanna", "Guinean mangroves": "Mangroves",
    "Guinean montane forests": "Montane forests", "West Sudanian savanna": "Sudanian savanna",
    "Western Guinean lowland forests": "Lowland forests",
}

# Wall-to-wall Hansen table (every 30 m pixel of Guinea): one row per ecoregion polygon x 1° cell x loss year
hansen = pd.read_parquet(HANSEN_LOSS_PARQUET)
for column in [c for c in hansen.columns if c.endswith("_km2")]:
    hansen[column.replace("_km2", "_ha")] = hansen.pop(column) * HA_PER_KM2
hansen_long = hansen.melt(
    id_vars=["ECO_NAME", "polygon_index", "cell_id", "lossyear", "year"],
    value_vars=[f"forest2000_gt{t}_ha" for t in CANOPY_THRESHOLDS], var_name="threshold", value_name="area_ha",
)
hansen_long["threshold"] = hansen_long["threshold"].str.extract(r"gt(\d+)_ha", expand=False).astype(int)

LAND_HA = hansen["land_ha"].sum()
print(f"Guinea land area: {LAND_HA:,.0f} ha")

### Map helpers

All maps share the same frame: Guinea's outline (thick) and the ecoregion borders (thin).

In [ ]:
WEST, SOUTH, EAST, NORTH = (np.floor(guinea_boundary.total_bounds * 10) / 10 + np.array([-0.1, -0.1, 0.2, 0.2])).tolist()


def map_frame(ax, title=None, extent=None):
    ecoregions.boundary.plot(ax=ax, color="#555555", linewidth=0.4)
    guinea_boundary.boundary.plot(ax=ax, color="black", linewidth=0.9)
    west, south, east, north = extent or (WEST, SOUTH, EAST, NORTH)
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_aspect(1 / np.cos(np.radians((south + north) / 2)))  # degrees look like distances at Guinea's latitude
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    for side in ax.spines.values():
        side.set_visible(False)
    if title:
        ax.set_title(title, fontsize=11, loc="left")


def colormap(name, bad="white"):
    cmap = plt.get_cmap(name).copy()
    cmap.set_bad(bad)
    return cmap

## 0. Data and methods

### 0.1 Hansen tables already computed

| Table | Made by | Content |
|---|---|---|
| `hansen_loss_by_polygon_cell_year.parquet` | `deforestation_hansen.ipynb` | every 30 m pixel of Guinea grouped by ecoregion polygon x 1° cell x loss year, forest 2000 at canopy > 0/10/30/50 % |
| `hansen_loss_by_release_year.parquet` | `hansen_annual_maps.ipynb` | loss per year in Guinea (canopy > 30 %) as published by each of the 12 Hansen releases (2015 → 2025) |
| `lulc_benchmark/*_h3res7.parquet` | `lulc_benchmark.ipynb` | 10 products read at 56,712 sample points |

The resolution-6 hexagons of `deforestation_hansen.ipynb` (~3,000 ha each) are too coarse to see where loss
happens, so a finer hexagon table is made below.

### 0.2 Choosing the H3 resolution

The target is hexagons of about 100–250 ha (1–2.5 km²). The table gives each H3 resolution at Guinea's latitude.

In [ ]:
rows = []
for resolution in range(6, 11):
    cell = h3.latlng_to_cell(10.4, -11, resolution)
    area_ha = np.mean([h3.cell_area(c, unit="m^2") for c in h3.grid_disk(cell, 3)]) / M2_PER_HA
    edge_m = np.mean([h3.edge_length(e, unit="m") for e in h3.origin_to_directed_edges(cell)])
    rows.append({
        "resolution": resolution,
        "hexagon_area_ha": area_ha,
        "width_m": 2 * edge_m,
        "hansen_pixels_per_hexagon": area_ha * M2_PER_HA / 760,   # a Hansen pixel is ~760 m² in Guinea
        "hexagons_in_guinea": LAND_HA / area_ha,
    })
resolutions = pd.DataFrame(rows).set_index("resolution")
resolutions.round(1)

**Choice: resolution 8 (~62 ha, ~1 km across).**

- **Closest to the 1 km² target.** Resolution 7 (~434 ha) is 4x coarser than asked; resolution 9 (~9 ha) is only a
  few clearings.
- **Enough pixels per hexagon.** ~800 Hansen pixels, so the share of forest lost in a hexagon is a stable number
  (at resolution 9, one 1-ha clearing is already 11 % of the hexagon).
- **Matches what a map can show.** On a map of the whole country one screen pixel is ~1 km: resolution 8 is the
  finest grid that is still visible; resolution 9 would only add detail for zoomed-in maps.
- **Manageable size.** ~390,000 hexagons (a few MB of parquet); resolution 9 would be 2.7 million.

Coarser resolutions (7, 6) are obtained afterwards by adding up children, without new computation.

### 0.3 Forest and loss per resolution-8 hexagon (one Earth Engine run, cached)

Sending ~390,000 hexagon outlines to Earth Engine is slow and expensive (a test on a 0.5° chunk: 145
EECU-seconds, ~3.5 EECU-hours for Guinea). Instead:

1. Earth Engine adds up the Hansen pixels on a regular grid of **0.00125°** cells. The grid is aligned with Hansen's
   0.00025° pixels, so each cell is **exactly 5 x 5 Hansen pixels** (~1.9 ha) and the sums are exact. For each cell:
   land area, forest 2000 at canopy > 10/30/50 %, and loss (canopy > 30 %) in 2001–2012, 2013–2019 and 2020–2025.
2. The grid is downloaded tile by tile (1° tiles, `ee.data.computePixels`). The same 0.5° test cost 19 EECU-seconds,
   7x less, so all of Guinea is ~0.8 EECU-hours. Tiles are cached as `.npz` (git-ignored) so an interrupted run resumes.
3. Locally, every grid cell is given to the hexagon containing its centre, and the hexagons are summed. Totals are
   exact; at the edge of a hexagon a 1.9 ha cell can fall on the wrong side, a small error for a 62 ha hexagon.

Only pixels inside the ecoregion polygons (Guinea) are counted, as in `deforestation_hansen.ipynb`, so the totals can
be compared in 0.4.

In [ ]:
GRID_BANDS = ["land", "forest2000_gt10", "forest2000_gt30", "forest2000_gt50",
              "loss_2001_2012", "loss_2013_2019", "loss_2020_2025"]


def connect_earth_engine():
    key_info = json.loads(KEY_FILE.read_text())
    credentials = ee.ServiceAccountCredentials(key_info["client_email"], key_file=str(KEY_FILE))
    ee.Initialize(credentials=credentials, project=key_info["project_id"])


def grid_image():
    """Earth Engine image on the 0.00125° grid: m² of each band in every grid cell."""
    hansen_image = ee.Image(HANSEN_ASSET)
    tree_cover = hansen_image.select("treecover2000")
    loss_year = hansen_image.select("lossyear").unmask(0)    # masked where there is no loss
    polygons_fc = ee.FeatureCollection([ee.Feature(ee.Geometry(g.__geo_interface__)) for g in polygons.geometry])
    inside_guinea = ee.Image(0).paint(polygons_fc, 1).reproject(hansen_image.projection())
    land = hansen_image.select("datamask").eq(1).And(inside_guinea)
    forest30 = tree_cover.gt(30).And(land)
    fractions = ee.Image.cat([
        land,
        tree_cover.gt(10).And(land),
        forest30,
        tree_cover.gt(50).And(land),
        forest30.And(loss_year.gte(1)).And(loss_year.lte(12)),
        forest30.And(loss_year.gte(13)).And(loss_year.lte(19)),
        forest30.And(loss_year.gte(20)),
    ]).rename(GRID_BANDS).toFloat()
    # mean of 25 Hansen pixels x area of the grid cell = sum of the 25 pixel areas (m², < 20,000 so uint16 fits)
    return fractions.reduceResolution(ee.Reducer.mean(), maxPixels=64).multiply(ee.Image.pixelArea()).round().toUint16()


def eecu_seconds_of(profile_ids):
    get_profiles = ee.ApiFunction.lookup("Profile.getProfiles").call
    for attempt in range(5):  # the profile is sometimes not ready on the server yet
        try:
            text = get_profiles(ids=profile_ids).getInfo()
        except ee.EEException:
            time.sleep(2 * (attempt + 1))
            continue
        return sum(float(line.split()[0]) for line in text.splitlines()[1:] if line.split() and line.split()[0] != "-")
    return None


def fetch_tile(image, west, south):
    """One 1° tile of the grid as a numpy structured array, plus (seconds, eecu_seconds)."""
    size = int(round(1 / GRID_DEG))
    request = {
        "expression": image, "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": size, "height": size}, "crsCode": "EPSG:4326",
                 "affineTransform": {"scaleX": GRID_DEG, "shearX": 0, "translateX": west,
                                     "shearY": 0, "scaleY": -GRID_DEG, "translateY": south + 1}},
    }
    profile_ids = []
    start = time.perf_counter()
    with ee.data.profiling(profile_ids.append):
        tile = ee.data.computePixels(request)
    return tile, time.perf_counter() - start, eecu_seconds_of(profile_ids)


def tile_to_hexagons(tile, west, south):
    """Sum the grid cells of a tile into H3 hexagons (by the hexagon containing each cell's centre)."""
    rows, cols = np.nonzero(tile["land"] > 0)
    lats = (south + 1) - (rows + 0.5) * GRID_DEG
    lons = west + (cols + 0.5) * GRID_DEG
    hexagon = [h3_int.latlng_to_cell(lat, lon, FINE_H3_RESOLUTION) for lat, lon in zip(lats.tolist(), lons.tolist())]
    sums = pd.DataFrame({band: tile[band][rows, cols].astype("int64") for band in GRID_BANDS})
    sums["h3"] = np.array(hexagon, dtype="int64")
    return sums.groupby("h3").sum()


def compute_fine_hexagons():
    connect_earth_engine()
    image = grid_image()
    guinea_shape = polygons.union_all()
    tiles = [(w, s) for w in range(int(np.floor(WEST)), int(np.ceil(EAST)))
             for s in range(int(np.floor(SOUTH)), int(np.ceil(NORTH))) if box(w, s, w + 1, s + 1).intersects(guinea_shape)]
    GRID_TILES_DIR.mkdir(exist_ok=True)
    parts, seconds, eecu = [], 0.0, 0.0
    for number, (west, south) in enumerate(tiles, start=1):
        path = GRID_TILES_DIR / f"tile_{west}_{south}.npz"
        if path.exists():
            tile = np.load(path)["tile"]
        else:
            tile, tile_seconds, tile_eecu = fetch_tile(image, west, south)
            np.savez_compressed(path, tile=tile)
            seconds, eecu = seconds + tile_seconds, eecu + (tile_eecu or 0)
            print(f"  tile {number}/{len(tiles)} ({west}, {south}): {tile_seconds:.0f} s, {tile_eecu} EECU-seconds")
            if tile_eecu and number == 1 and tile_eecu * len(tiles) / 3600 > MAX_EECU_HOURS:
                raise RuntimeError(f"projected {tile_eecu * len(tiles) / 3600:.1f} EECU-hours > MAX_EECU_HOURS")
        parts.append(tile_to_hexagons(tile, west, south))
    sums = pd.concat(parts).groupby(level=0).sum() / M2_PER_HA  # hexagons on a tile edge are in two tiles
    sums.columns = [f"{band}_ha" for band in GRID_BANDS]
    run_info = {"computed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "hansen_asset": HANSEN_ASSET,
                "tiles": len(tiles), "grid_deg": GRID_DEG, "seconds": round(seconds, 1),
                "eecu_hours": round(eecu / 3600, 3)}
    return sums, run_info


def add_location(sums):
    """Hexagon ID, centre and ecoregion (the one containing the centre, or the nearest one at the border)."""
    table = sums.reset_index(names="h3")
    table.insert(0, "h3_id", [h3.int_to_str(int(cell)) for cell in table["h3"]])
    table["lat"], table["lon"] = zip(*(h3_int.cell_to_latlng(int(cell)) for cell in table["h3"]))
    centres = gpd.GeoDataFrame(table[["h3"]], geometry=gpd.points_from_xy(table["lon"], table["lat"]), crs="EPSG:4326")
    utm = centres.estimate_utm_crs()
    nearest = gpd.sjoin_nearest(centres.to_crs(utm), polygons[["ECO_NAME", "geometry"]].to_crs(utm), how="left")
    table["ECO_NAME"] = nearest.groupby(level=0)["ECO_NAME"].first()
    return table.drop(columns="h3")


if H3_FINE_PARQUET.exists() and not FORCE_RECOMPUTE:
    fine = pd.read_parquet(H3_FINE_PARQUET)
    print(f"Read {H3_FINE_PARQUET.name} (no Earth Engine compute used): {json.loads(H3_FINE_RUN_JSON.read_text())}")
else:
    sums, run_info = compute_fine_hexagons()
    fine = add_location(sums)
    fine.to_parquet(H3_FINE_PARQUET, index=False)
    H3_FINE_RUN_JSON.write_text(json.dumps(run_info, indent=2))
    print(f"Computed and saved {H3_FINE_PARQUET.name}: {run_info}")

fine["h3"] = [h3.str_to_int(cell) for cell in fine["h3_id"]]
fine["loss_2013_2025_ha"] = fine["loss_2013_2019_ha"] + fine["loss_2020_2025_ha"]
fine["loss_2001_2025_ha"] = fine["loss_2001_2012_ha"] + fine["loss_2013_2025_ha"]
fine["forest_2012_ha"] = fine["forest2000_gt30_ha"] - fine["loss_2001_2012_ha"]   # forest at the start of 2013
fine["forest_2025_ha"] = fine["forest2000_gt30_ha"] - fine["loss_2001_2025_ha"]
print(f"{len(fine):,} hexagons of resolution {FINE_H3_RESOLUTION}, mean land {fine['land_ha'].mean():.1f} ha")

### 0.4 Check: the hexagon table against the wall-to-wall table

Both count the same pixels, so the totals must match.

In [ ]:
forest30 = hansen["forest2000_gt30_ha"]
in_period = lambda first, last: hansen["year"].between(first, last)
checks = pd.DataFrame({
    "wall_to_wall_ha": [
        hansen["land_ha"].sum(), hansen["forest2000_gt10_ha"].sum(), forest30.sum(), hansen["forest2000_gt50_ha"].sum(),
        forest30[in_period(2001, 2012)].sum(), forest30[in_period(2013, 2019)].sum(), forest30[in_period(2020, 2025)].sum(),
    ],
    "hexagons_ha": [fine[f"{band}_ha"].sum() for band in GRID_BANDS],
}, index=GRID_BANDS)
checks["difference_%"] = 100 * (checks["hexagons_ha"] / checks["wall_to_wall_ha"] - 1)
checks.round(2)

### Hexagon map helper

Resolution-8 hexagons are drawn as an image: every point of a 0.0025° display grid (~275 m) takes the value of the
hexagon it falls in. This draws 390,000 hexagons in a second.

In [ ]:
DISPLAY_DEG = 0.0025
display_lons = np.arange(WEST + DISPLAY_DEG / 2, EAST, DISPLAY_DEG)
display_lats = np.arange(NORTH - DISPLAY_DEG / 2, SOUTH, -DISPLAY_DEG)
lon_grid, lat_grid = np.meshgrid(display_lons, display_lats)
display_cells = np.array([h3_int.latlng_to_cell(lat, lon, FINE_H3_RESOLUTION)
                          for lat, lon in zip(lat_grid.ravel().tolist(), lon_grid.ravel().tolist())], dtype="int64")
display_position = pd.Index(fine["h3"]).get_indexer(display_cells)  # row of `fine` for every display point, -1 = none


def hexagon_image(values):
    """2-D image of one value per row of `fine` (NaN outside Guinea)."""
    values = np.asarray(values, dtype=float)
    image = np.where(display_position >= 0, values[display_position], np.nan)
    return image.reshape(lat_grid.shape)


def hexagon_map(ax, values, cmap, vmin, vmax, title, colorbar_label, extent=None):
    shown = ax.imshow(hexagon_image(values), extent=[WEST, EAST, SOUTH, NORTH], cmap=cmap, vmin=vmin, vmax=vmax,
                      interpolation="nearest")
    map_frame(ax, title, extent)
    plt.colorbar(shown, ax=ax, shrink=0.7, label=colorbar_label)

# Part I – Hansen Global Forest Change

## 1. Sensitivity to the canopy threshold

Hansen does not say what a forest is: it gives the canopy cover of every pixel in 2000 (0–100 %), and the user picks
a threshold. The same data give very different forest areas depending on that choice.

In [ ]:
loss_rows = hansen_long[hansen_long["lossyear"] > 0]
threshold_summary = pd.DataFrame({
    "forest_2000_ha": hansen_long.groupby("threshold")["area_ha"].sum(),
    "loss_2001_2025_ha": loss_rows.groupby("threshold")["area_ha"].sum(),
    f"loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_ha": loss_rows[loss_rows["year"] >= METHOD_CHANGE_YEAR].groupby("threshold")["area_ha"].sum(),
})
threshold_summary.insert(1, "forest_%_of_guinea", 100 * threshold_summary["forest_2000_ha"] / LAND_HA)
threshold_summary["forest_vs_30%"] = threshold_summary["forest_2000_ha"] / threshold_summary.loc[CANOPY_THRESHOLD, "forest_2000_ha"]
threshold_summary.index = [f"> {t} %" for t in threshold_summary.index]
threshold_summary.round(2)

### 1.1 Guinea

In [ ]:
labels = [f"> {t} %" for t in CANOPY_THRESHOLDS]
colors = [THRESHOLD_COLORS[t] for t in CANOPY_THRESHOLDS]
fig = make_subplots(rows=1, cols=2, subplot_titles=["Forest in 2000", f"Loss {METHOD_CHANGE_YEAR}–{LAST_YEAR}"])
fig.add_trace(go.Bar(x=labels, y=threshold_summary["forest_2000_ha"], marker_color=colors,
                     customdata=threshold_summary["forest_%_of_guinea"],
                     hovertemplate="%{x}: %{y:,.0f} ha (%{customdata:.1f} % of Guinea)<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=threshold_summary[f"loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_ha"], marker_color=colors,
                     hovertemplate="%{x}: %{y:,.0f} ha<extra></extra>"), row=1, col=2)
fig.update_yaxes(title_text="Area (ha)", tickformat=",.0f")
fig.update_xaxes(title_text="Canopy cover in 2000")
fig.update_layout(title="Hansen forest area and loss in Guinea for different canopy thresholds",
                  showlegend=False, template=PLOT_TEMPLATE, height=450)
fig.show()

### 1.2 Per ecoregion

In [ ]:
by_ecoregion = pd.concat({
    "Forest in 2000": hansen_long.groupby(["ECO_NAME", "threshold"])["area_ha"].sum(),
    f"Loss {METHOD_CHANGE_YEAR}–{LAST_YEAR}": loss_rows[loss_rows["year"] >= METHOD_CHANGE_YEAR].groupby(["ECO_NAME", "threshold"])["area_ha"].sum(),
}, names=["quantity"]).rename("area_ha").reset_index()
by_ecoregion["canopy"] = "> " + by_ecoregion["threshold"].astype(str) + " %"

fig = px.bar(
    by_ecoregion, x="ECO_NAME", y="area_ha", color="canopy", barmode="group", facet_row="quantity",
    color_discrete_map={f"> {t} %": c for t, c in THRESHOLD_COLORS.items()},
    labels={"area_ha": "Area (ha)", "ECO_NAME": "", "canopy": "Canopy in 2000"},
    title="Forest area and loss per ecoregion for different canopy thresholds", template=PLOT_TEMPLATE, height=750,
)
fig.update_yaxes(matches=None, tickformat=",.0f")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_traces(hovertemplate="%{y:,.0f} ha")
fig.show()

(by_ecoregion.pivot_table(index=["quantity", "ECO_NAME"], columns="canopy", values="area_ha")
 [[f"> {t} %" for t in CANOPY_THRESHOLDS]].round(0))

### 1.3 Forest cover in 2000 at 10, 30 and 50 % canopy

Share of each resolution-8 hexagon (~62 ha) that is forest at each threshold. At 10 % almost all of Guinea is
"forest", including the wooded savanna of the north; at 50 % only the dense forests of the south-east and the
Fouta Djallon valleys remain.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, threshold in zip(axes, [10, 30, 50]):
    share = 100 * fine[f"forest2000_gt{threshold}_ha"] / fine["land_ha"]
    total = fine[f"forest2000_gt{threshold}_ha"].sum()
    hexagon_map(ax, share, colormap("Greens"), 0, 100,
                f"Canopy > {threshold} %: {total:,.0f} ha ({100 * total / LAND_HA:.0f} % of Guinea)", "Forest (% of hexagon)")
fig.suptitle("Hansen forest cover in 2000, H3 resolution 8", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Forest area over time and the 2013 method change

Forest at the end of each year = forest in 2000 (canopy > 30 %) minus all loss up to that year. Hansen does not add
regrowth after 2012, so this can only go down.

In [ ]:
national_loss = (hansen[hansen["lossyear"] > 0].groupby("year")["forest2000_gt30_ha"].sum()
                 .reindex(range(FIRST_YEAR, LAST_YEAR + 1), fill_value=0))
forest_2000_ha = hansen["forest2000_gt30_ha"].sum()
forest_by_year = pd.concat([pd.Series({2000: forest_2000_ha}), forest_2000_ha - national_loss.cumsum()])


def period_of(year):
    if year < METHOD_CHANGE_YEAR:
        return f"{FIRST_YEAR}–{METHOD_CHANGE_YEAR - 1}"
    return f"{METHOD_CHANGE_YEAR}–{RECENT_YEAR - 1}" if year < RECENT_YEAR else f"{RECENT_YEAR}–{LAST_YEAR}"


fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.55, 0.45],
                    subplot_titles=["Forest area (canopy > 30 % in 2000, minus loss)", "Annual loss"])
fig.add_trace(go.Scatter(x=forest_by_year.index, y=forest_by_year, mode="lines+markers", line={"color": "#2e7d32"},
                         name="Forest", hovertemplate="End of %{x}: %{y:,.0f} ha<extra></extra>"), row=1, col=1)
for period, color in PERIOD_COLORS.items():
    years = [y for y in national_loss.index if period_of(y) == period]
    fig.add_trace(go.Bar(x=years, y=national_loss[years], marker_color=color, name=period,
                         hovertemplate="%{x}: %{y:,.0f} ha<extra></extra>"), row=2, col=1)
for row in (1, 2):
    fig.add_vline(x=METHOD_CHANGE_YEAR - 0.5, line_dash="dash", line_color="grey", row=row, col=1)
fig.add_annotation(x=METHOD_CHANGE_YEAR - 0.5, y=1, yref="paper", text="Hansen method change", showarrow=False,
                   xanchor="left", yanchor="bottom", font={"color": "grey"})
fig.update_yaxes(title_text="ha", tickformat=",.0f")
fig.update_layout(title="Forest area and annual loss in Guinea (Hansen)", template=PLOT_TEMPLATE, height=650,
                  legend_title_text="Period")
fig.show()

period_rows = {}
for period in [*PERIOD_COLORS, f"{METHOD_CHANGE_YEAR}–{LAST_YEAR}"]:
    first, last = (int(year) for year in period.split("–"))
    period_loss = national_loss.loc[first:last]
    period_rows[period] = {"years": len(period_loss), "total_loss_ha": period_loss.sum(),
                           "mean_annual_loss_ha": period_loss.mean()}
periods = pd.DataFrame(period_rows).T
periods["mean_annual_loss_%_of_2000_forest"] = 100 * periods["mean_annual_loss_ha"] / forest_2000_ha
periods.round(2)

### 2.1 The break is in every release

Each yearly Hansen release adds one year of loss. The lines below are the loss per year as published by each of
the 12 releases still on Earth Engine (2015 → 2025): they lie on top of each other, so **releases do not reprocess
earlier years**, and every release shows the same jump in 2013. The years before 2013 were mapped with the original
2013 method and Landsat 5/7 only; from 2013 on, Landsat 8 and an improved model detect much more small-scale loss.
The 2001–2012 values are therefore not comparable with the later ones.

In [ ]:
releases = pd.read_parquet(HANSEN_RELEASES_PARQUET)
releases = releases[releases["lossyear"] > 0].assign(year=lambda df: 2000 + df["lossyear"],
                                                     loss_ha=lambda df: df["area_km2"] * HA_PER_KM2)
release_names = list(dict.fromkeys(releases["release"]))
release_colors = px.colors.sample_colorscale("Viridis", np.linspace(0, 0.9, len(release_names)))
fig = px.line(releases, x="year", y="loss_ha", color="release", markers=True,
              color_discrete_map=dict(zip(release_names, release_colors)),
              labels={"loss_ha": "Loss (ha)", "year": "Year", "release": "Release"},
              title="Annual loss in Guinea as published by each Hansen release (canopy > 30 %)", template=PLOT_TEMPLATE)
fig.add_vline(x=METHOD_CHANGE_YEAR - 0.5, line_dash="dash", line_color="grey")
fig.update_yaxes(tickformat=",.0f")
fig.update_traces(hovertemplate="%{x}: %{y:,.0f} ha")
fig.show()

### 2.2 Adopted for the analysis

- **Forest = canopy > 30 % in 2000** (Global Forest Watch convention; close to FAO FRA 2020's 6.19 million ha, see
  Part II). The legal definition (> 10 %) is discussed in `lulc_benchmark.ipynb`.
- **Only 2013 onwards** is analysed (consistent method), with a second focus on **2020 onwards** (the most recent,
  and the period of the land-cover benchmark).

## 3. Where the forest is

Forest area per ecoregion and per resolution-8 hexagon, at the start of the analysis (2013 = forest in 2000 minus the
2001–2012 loss) and in 2025.

In [ ]:
eco_forest = pd.DataFrame({
    "land_ha": hansen.groupby("ECO_NAME")["land_ha"].sum(),
    "forest_2000_ha": hansen.groupby("ECO_NAME")["forest2000_gt30_ha"].sum(),
    "loss_2001_2012_ha": hansen[hansen["year"].between(FIRST_YEAR, METHOD_CHANGE_YEAR - 1)].groupby("ECO_NAME")["forest2000_gt30_ha"].sum(),
    "loss_2013_2019_ha": hansen[hansen["year"].between(METHOD_CHANGE_YEAR, RECENT_YEAR - 1)].groupby("ECO_NAME")["forest2000_gt30_ha"].sum(),
    "loss_2020_2025_ha": hansen[hansen["year"].between(RECENT_YEAR, LAST_YEAR)].groupby("ECO_NAME")["forest2000_gt30_ha"].sum(),
})
eco_forest["forest_2012_ha"] = eco_forest["forest_2000_ha"] - eco_forest["loss_2001_2012_ha"]
eco_forest["forest_2019_ha"] = eco_forest["forest_2012_ha"] - eco_forest["loss_2013_2019_ha"]
eco_forest["forest_2025_ha"] = eco_forest["forest_2019_ha"] - eco_forest["loss_2020_2025_ha"]
eco_forest["forest_cover_2025_%"] = 100 * eco_forest["forest_2025_ha"] / eco_forest["land_ha"]
eco_forest["share_of_guinea_forest_2025_%"] = 100 * eco_forest["forest_2025_ha"] / eco_forest["forest_2025_ha"].sum()

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
eco_map = ecoregions.merge(eco_forest.reset_index(), on="ECO_NAME")
eco_map.plot(column="forest_2025_ha", cmap="Greens", ax=axes[0], edgecolor="#555555", linewidth=0.4,
             legend=True, legend_kwds={"label": "Forest in 2025 (ha)", "shrink": 0.7, "format": "{x:,.0f}"})
map_frame(axes[0], "Forest per ecoregion, 2025 (canopy > 30 %)")
for _, row in eco_map.iterrows():
    biggest_part = max(getattr(row.geometry, "geoms", [row.geometry]), key=lambda part: part.area)
    point = biggest_part.representative_point()
    axes[0].annotate(f"{ECOREGION_SHORT[row['ECO_NAME']]}\n{row['forest_2025_ha']:,.0f} ha\n({row['forest_cover_2025_%']:.0f} % cover)",
                     (point.x, point.y), ha="center", fontsize=8,
                     bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8, "linewidth": 0})
hexagon_map(axes[1], fine["forest_2025_ha"], colormap("Greens"), 0, fine["land_ha"].median(),
            f"Forest per hexagon (~62 ha), 2025: {fine['forest_2025_ha'].sum():,.0f} ha", "Forest (ha per hexagon)")
plt.tight_layout()
plt.show()

In [ ]:
forest_years = eco_forest[["forest_2000_ha", "forest_2012_ha", "forest_2019_ha", "forest_2025_ha"]].reset_index().melt(
    id_vars="ECO_NAME", var_name="year", value_name="forest_ha")
forest_years["year"] = forest_years["year"].str.extract(r"(\d{4})", expand=False).map(
    {"2000": "2000", "2012": "Start of 2013", "2019": "End of 2019", "2025": "End of 2025"})
fig = px.bar(forest_years, x="ECO_NAME", y="forest_ha", color="year", barmode="group",
             color_discrete_sequence=["#c7e9c0", "#74c476", "#238b45", "#00441b"],
             labels={"forest_ha": "Forest (ha)", "ECO_NAME": "", "year": ""},
             title="Forest area per ecoregion (Hansen, canopy > 30 %)", template=PLOT_TEMPLATE)
fig.update_yaxes(tickformat=",.0f")
fig.update_traces(hovertemplate="%{y:,.0f} ha")
fig.show()
eco_forest.round(1)

## 4. Which ecoregions lose their forest fastest?

Annual loss rate = loss in the year / forest at the start of that year (forest 2000 minus all earlier loss). Unlike
hectares, the rate compares ecoregions of very different size.

In [ ]:
eco_year = (hansen[hansen["lossyear"] > 0].groupby(["ECO_NAME", "year"])["forest2000_gt30_ha"].sum()
            .unstack(fill_value=0).reindex(columns=range(FIRST_YEAR, LAST_YEAR + 1), fill_value=0))
eco_year.loc["Guinea"] = eco_year.sum()
forest_start = pd.concat([eco_forest["forest_2000_ha"], pd.Series({"Guinea": eco_forest["forest_2000_ha"].sum()})])
forest_at_start_of_year = eco_year.cumsum(axis=1).shift(1, axis=1, fill_value=0).rsub(forest_start, axis=0)
loss_rate = 100 * eco_year / forest_at_start_of_year

rate_long = (loss_rate.loc[:, METHOD_CHANGE_YEAR:].stack().rename("loss_rate_%").to_frame()
             .join(eco_year.loc[:, METHOD_CHANGE_YEAR:].stack().rename("loss_ha"))
             .rename_axis(["ECO_NAME", "year"]).reset_index())
fig = px.line(rate_long, x="year", y="loss_rate_%", color="ECO_NAME", markers=True, custom_data=["loss_ha"],
              color_discrete_map={**ECOREGION_COLORS, "Guinea": "black"},
              labels={"loss_rate_%": "Loss (% of the forest at the start of the year)", "year": "Year", "ECO_NAME": ""},
              title=f"Annual forest loss rate per ecoregion, {METHOD_CHANGE_YEAR}–{LAST_YEAR}", template=PLOT_TEMPLATE)
fig.update_traces(hovertemplate="%{x}: %{y:.2f} % (%{customdata[0]:,.0f} ha)")
fig.update_traces(line={"dash": "dash", "width": 3}, selector={"name": "Guinea"})
fig.add_vline(x=RECENT_YEAR - 0.5, line_dash="dot", line_color="grey")
fig.show()

mean_rates = pd.DataFrame({
    f"mean_loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_ha_per_year": eco_year.loc[:, METHOD_CHANGE_YEAR:].mean(axis=1),
    f"mean_rate_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_%": loss_rate.loc[:, METHOD_CHANGE_YEAR:].mean(axis=1),
    f"mean_loss_{RECENT_YEAR}_{LAST_YEAR}_ha_per_year": eco_year.loc[:, RECENT_YEAR:].mean(axis=1),
    f"mean_rate_{RECENT_YEAR}_{LAST_YEAR}_%": loss_rate.loc[:, RECENT_YEAR:].mean(axis=1),
})
rate_bars = mean_rates.filter(like="rate").reset_index(names="ECO_NAME").melt(id_vars="ECO_NAME", var_name="period", value_name="rate")
rate_bars["period"] = rate_bars["period"].str.extract(r"(\d{4}_\d{4})", expand=False).str.replace("_", "–")
fig = px.bar(rate_bars, x="ECO_NAME", y="rate", color="period", barmode="group",
             color_discrete_sequence=["#e67e22", "#c0392b"],
             labels={"rate": "Mean annual loss rate (%)", "ECO_NAME": "", "period": "Period"},
             title="Mean annual loss rate per period", template=PLOT_TEMPLATE)
fig.update_traces(hovertemplate="%{y:.2f} % per year")
fig.show()
mean_rates.round(2)

## 5. Where the loss happens: resolution-8 hexagons

Cumulative loss per hexagon (~62 ha) since the method change. Hexagons with less than 5 ha of forest at the start
of 2013 are left white in the rate map (a few hectares of loss there would show as a very high %).

In [ ]:
MIN_FOREST_HA = 5
rate_2013 = (100 * fine["loss_2013_2025_ha"] / fine["forest_2012_ha"]).where(fine["forest_2012_ha"] >= MIN_FOREST_HA)

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
hexagon_map(axes[0], fine["loss_2013_2025_ha"], colormap("OrRd"), 0, fine["loss_2013_2025_ha"].quantile(0.99),
            f"Loss {METHOD_CHANGE_YEAR}–{LAST_YEAR}: {fine['loss_2013_2025_ha'].sum():,.0f} ha", "Loss (ha per hexagon)")
hexagon_map(axes[1], rate_2013, colormap("OrRd"), 0, 100,
            f"Loss {METHOD_CHANGE_YEAR}–{LAST_YEAR} as % of the 2013 forest", "Loss (% of forest)")
hexagon_map(axes[2], fine["loss_2020_2025_ha"], colormap("OrRd"), 0, fine["loss_2020_2025_ha"].quantile(0.99),
            f"Loss {RECENT_YEAR}–{LAST_YEAR}: {fine['loss_2020_2025_ha'].sum():,.0f} ha", "Loss (ha per hexagon)")
fig.suptitle("Hansen forest loss per H3 resolution-8 hexagon (canopy > 30 %)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. How concentrated is the loss?

Hexagons are sorted from the most to the least loss (2013–2025) and accumulated: the curve gives the share of
Guinea's loss that falls in a given share of its land. The same curve is drawn for resolution 7 and 6 (children added
up) to show that the answer depends on the size of the zones: smaller zones isolate the hotspots better.

Part of the concentration is trivial: 41 % of Guinea (mostly the northern savanna) had no forest loss because it has
little forest. The dashed curve therefore measures the same thing against the **forest** area at the start of 2013
instead of the land: it tells whether loss is concentrated *within the forest*.

In [ ]:
def concentration_curve(table, loss_column, basis_column="land_ha"):
    """Share of the loss (y) in the share of the basis area (x), hexagons sorted from most to least loss."""
    ordered = table.sort_values(loss_column, ascending=False)
    return pd.DataFrame({
        "share_of_land_%": 100 * ordered[basis_column].cumsum() / ordered[basis_column].sum(),
        "share_of_loss_%": 100 * ordered[loss_column].cumsum() / ordered[loss_column].sum(),
    })


def at_share_of_land(curve, land_share):
    return np.interp(land_share, curve["share_of_land_%"], curve["share_of_loss_%"])


def aggregate_to(resolution):
    parent = [h3_int.cell_to_parent(int(cell), resolution) for cell in fine["h3"]]
    return fine.groupby(parent)[["land_ha", "forest_2012_ha", "loss_2013_2025_ha", "loss_2020_2025_ha"]].sum()


LAND_SHARES = [10, 25, 40, 50]
curves, rows = {}, []
for resolution, table in [(FINE_H3_RESOLUTION, fine), (7, aggregate_to(7)), (6, aggregate_to(6))]:
    for basis_column, basis in [("land_ha", "land"), ("forest_2012_ha", "forest 2013")]:
        for loss_column, period in [("loss_2013_2025_ha", f"{METHOD_CHANGE_YEAR}–{LAST_YEAR}"),
                                    ("loss_2020_2025_ha", f"{RECENT_YEAR}–{LAST_YEAR}")]:
            curve = concentration_curve(table, loss_column, basis_column)
            curves[(resolution, basis, period)] = curve
            rows.append({"resolution": resolution, "share_of": basis, "period": period,
                         **{f"loss_in_top_{s}%": at_share_of_land(curve, s) for s in LAND_SHARES}})
concentration = pd.DataFrame(rows).set_index(["resolution", "share_of", "period"])

fig = go.Figure()
for (resolution, basis, period), curve in curves.items():
    if period != f"{METHOD_CHANGE_YEAR}–{LAST_YEAR}" or (basis != "land" and resolution != FINE_H3_RESOLUTION):
        continue
    thin = curve.iloc[np.linspace(0, len(curve) - 1, 2000).astype(int)]  # 2,000 points are enough to draw it
    fig.add_trace(go.Scatter(x=thin["share_of_land_%"], y=thin["share_of_loss_%"], mode="lines",
                             name=f"H3 res {resolution}, share of {basis}",
                             line={"width": 3 if resolution == FINE_H3_RESOLUTION else 1.5,
                                   "dash": "dash" if basis != "land" else "solid"},
                             hovertemplate=f"%{{x:.0f}} % of {basis} → %{{y:.0f}} % of loss<extra></extra>"))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line={"dash": "dash", "color": "grey"},
                         name="Loss spread evenly", hoverinfo="skip"))
fine_curve = curves[(FINE_H3_RESOLUTION, "land", f"{METHOD_CHANGE_YEAR}–{LAST_YEAR}")]
for share in LAND_SHARES:
    value = at_share_of_land(fine_curve, share)
    fig.add_annotation(x=share, y=value, text=f"{share} % of land → {value:.0f} % of loss", showarrow=True,
                       arrowhead=2, ax=60, ay=25)
fig.update_layout(title=f"Concentration of forest loss {METHOD_CHANGE_YEAR}–{LAST_YEAR}",
                  xaxis_title="Share of Guinea's land, or forest (%), hexagons sorted by loss", yaxis_title="Share of the loss (%)",
                  template=PLOT_TEMPLATE, height=550)
fig.show()
concentration.round(1)

## 7. Hotspots

Each hexagon is classified by where it falls on the concentration curve (2013–2025, resolution 8):

- **Core hotspots**: the hexagons with the most loss that together hold **50 %** of Guinea's loss;
- **Hotspots**: the next ones, up to **80 %** of the loss;
- **Other loss**: the remaining 20 % of the loss, spread over the rest.

The right-hand map zooms on the densest cluster of core hotspots.

In [ ]:
ordered = fine.sort_values("loss_2013_2025_ha", ascending=False)
cumulative_share = 100 * ordered["loss_2013_2025_ha"].cumsum() / ordered["loss_2013_2025_ha"].sum()
HOTSPOT_CLASSES = ["No loss", "Other loss", "Hotspot (50–80 % of loss)", "Core hotspot (first 50 % of loss)"]
fine["hotspot_class"] = pd.Series(
    np.select([cumulative_share <= 50, cumulative_share <= 80, ordered["loss_2013_2025_ha"] > 0], [3, 2, 1], 0),
    index=ordered.index).reindex(fine.index)
hotspot_cmap = ListedColormap(["#f0f0f0", "#fdd49e", "#ef6548", "#7f0000"])
hotspot_cmap.set_bad("white")

# Densest cluster: the resolution-5 zone (~25,000 ha) with the most core-hotspot loss
core = fine[fine["hotspot_class"] == 3]
zone = pd.Series([h3_int.cell_to_parent(int(cell), 5) for cell in core["h3"]], index=core.index)
best_zone = core.groupby(zone)["loss_2013_2025_ha"].sum().idxmax()
zone_lat, zone_lon = h3_int.cell_to_latlng(best_zone)
ZOOM = (zone_lon - 0.45, zone_lat - 0.35, zone_lon + 0.45, zone_lat + 0.35)

fig, axes = plt.subplots(1, 2, figsize=(19, 7.5), gridspec_kw={"width_ratios": [1.35, 1]})
for ax, extent, title in [(axes[0], None, f"Loss hotspots {METHOD_CHANGE_YEAR}–{LAST_YEAR}, H3 resolution 8"),
                          (axes[1], ZOOM, f"Zoom around {zone_lat:.2f}° N, {abs(zone_lon):.2f}° W")]:
    ax.imshow(hexagon_image(fine["hotspot_class"]), extent=[WEST, EAST, SOUTH, NORTH], cmap=hotspot_cmap,
              vmin=-0.5, vmax=3.5, interpolation="nearest")
    map_frame(ax, title, extent)
axes[0].add_patch(plt.Rectangle(ZOOM[:2], ZOOM[2] - ZOOM[0], ZOOM[3] - ZOOM[1], fill=False, edgecolor="blue", linewidth=1.5))
area_by_class = fine.groupby("hotspot_class")["land_ha"].sum()
axes[0].legend(handles=[Patch(color=hotspot_cmap(i), label=f"{name}: {100 * area_by_class.get(i, 0) / LAND_HA:.0f} % of land")
                        for i, name in reversed(list(enumerate(HOTSPOT_CLASSES)))], loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
hotspot_ecoregion = (fine.assign(hotspot=fine["hotspot_class"].map(dict(enumerate(HOTSPOT_CLASSES))))
                     .groupby(["ECO_NAME", "hotspot"])[["land_ha", "loss_2013_2025_ha"]].sum().reset_index())
fig = px.bar(hotspot_ecoregion[hotspot_ecoregion["hotspot"] != "No loss"], x="ECO_NAME", y="loss_2013_2025_ha",
             color="hotspot", color_discrete_map=dict(zip(HOTSPOT_CLASSES, ["#f0f0f0", "#fdd49e", "#ef6548", "#7f0000"])),
             category_orders={"hotspot": HOTSPOT_CLASSES[::-1]}, custom_data=["land_ha"],
             labels={"loss_2013_2025_ha": f"Loss {METHOD_CHANGE_YEAR}–{LAST_YEAR} (ha)", "ECO_NAME": "", "hotspot": ""},
             title="Where the loss of each ecoregion happens", template=PLOT_TEMPLATE)
fig.update_yaxes(tickformat=",.0f")
fig.update_traces(hovertemplate="%{y:,.0f} ha of loss in %{customdata[0]:,.0f} ha of land")
fig.show()

# The 15 resolution-6 zones (~3,000 ha) with the most loss, to check on imagery
zone6 = pd.Series([h3_int.cell_to_parent(int(cell), 6) for cell in fine["h3"]], index=fine.index)
top_zones = (fine.groupby(zone6).agg(main_ecoregion=("ECO_NAME", lambda s: s.mode().iloc[0]), land_ha=("land_ha", "sum"),
                                     forest_2012_ha=("forest_2012_ha", "sum"), loss_2013_2025_ha=("loss_2013_2025_ha", "sum"),
                                     loss_2020_2025_ha=("loss_2020_2025_ha", "sum"))
             .nlargest(15, "loss_2013_2025_ha"))
top_zones["loss_%_of_2013_forest"] = 100 * top_zones["loss_2013_2025_ha"] / top_zones["forest_2012_ha"]
top_zones["centre_lat"], top_zones["centre_lon"] = zip(*(h3_int.cell_to_latlng(int(cell)) for cell in top_zones.index))
top_zones.index = [h3.int_to_str(int(cell)) for cell in top_zones.index]
top_zones.round(2)

# Part II – Land-cover benchmark

## 8. Products

`lulc_benchmark.ipynb` read 10 products at the same points. This report uses:

| Product | Resolution | Year used | Role here |
|---|---|---|---|
| **Esri LULC** (Impact Observatory) | 10 m, Sentinel-2 | 2020 | land cover |
| **Dynamic World** (Google / WRI) | 10 m, Sentinel-2 | 2020 (most frequent label of the year) | land cover |
| **ESA WorldCover** v100 | 10 m, Sentinel-1/2 | 2020 | land cover |
| JRC GFC2020 (EUDR baseline) | 10 m | 2020 | forest only |
| GLC_FCS30D | 30 m, Landsat | 2020 | forest |
| Copernicus LC100 | 100 m, PROBA-V | 2019 (last year) | forest |
| Hansen GFC | 30 m, Landsat | 2020 (canopy > 30 % in 2000, minus loss) | forest |

Esri, Dynamic World and WorldCover are the three 10 m Sentinel-2 land-cover maps with a full legend, and are
treated as the most reliable for the non-forest classes. The four others are added for forest.

Each product's classes are translated to the common legend of `lulc_benchmark.ipynb` (section 3 there).

In [ ]:
NODATA = -1  # masked pixels, as saved by lulc_benchmark.ipynb
HARMONISED_CLASSES = {1: "Tree cover", 2: "Shrub & grassland", 3: "Cropland", 4: "Built-up",
                      5: "Bare / sparse vegetation", 6: "Water", 7: "Herbaceous wetland"}
HARMONISED_COLORS = {"Tree cover": "#2e7d32", "Shrub & grassland": "#c5b358", "Cropland": "#f39c12", "Built-up": "#c0392b",
                     "Bare / sparse vegetation": "#a1887f", "Water": "#2980b9", "Herbaceous wetland": "#7fcdbb"}

# Same definitions as lulc_benchmark.ipynb, section 3
PRODUCTS = {
    "esri": {"label": "Esri LULC", "year": 2020, "forest_classes": [2], "nodata": [0, 9, 10],
             "crosswalk": {1: 6, 2: 1, 4: 7, 5: 3, 7: 4, 8: 5, 11: 2}},
    "dynamic_world": {"label": "Dynamic World", "year": 2020, "forest_classes": [1], "nodata": [8],
                      "crosswalk": {0: 6, 1: 1, 2: 2, 3: 7, 4: 3, 5: 2, 6: 4, 7: 5}},
    "worldcover": {"label": "ESA WorldCover", "year": 2020, "forest_classes": [10, 95], "nodata": [0, 70],
                   "crosswalk": {10: 1, 20: 2, 30: 2, 40: 3, 50: 4, 60: 5, 80: 6, 90: 7, 95: 1, 100: 2}},
    "gfc2020": {"label": "JRC GFC2020", "year": 2020, "forest_classes": [1], "nodata": []},
    "glc_fcs30d": {"label": "GLC_FCS30D", "year": 2020, "nodata": [0, 220, 250, 255],
                   "forest_classes": [51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 185]},
    "cgls_lc100": {"label": "Copernicus LC100", "year": 2019, "nodata": [0, 70, 255],
                   "forest_classes": [*range(111, 117), *range(121, 127)]},
    "hansen": {"label": "Hansen GFC (> 30 %)", "year": 2020},
}
LULC_KEYS = ["esri", "dynamic_world", "worldcover"]
FOREST_KEYS = list(PRODUCTS)
PRODUCT_COLORS = {"Esri LULC": "#1f77b4", "Dynamic World": "#17becf", "ESA WorldCover": "#2ca02c",
                  "JRC GFC2020": "#9467bd", "GLC_FCS30D": "#8c564b", "Copernicus LC100": "#e377c2",
                  "Hansen GFC (> 30 %)": "#d62728", "Hansen GFC (> 10 %)": "#ff9896"}

points = pd.read_parquet(SAMPLE_POINTS_PARQUET).set_index("point_id")
weights_ha = points["weight_km2"] * HA_PER_KM2
TOTAL_HA = weights_ha.sum()

forest, lulc = {}, {}
for key, spec in PRODUCTS.items():
    values = pd.read_parquet(BENCHMARK_DIR / f"{key}_h3res7.parquet").set_index("point_id").reindex(points.index)
    if key == "hansen":
        lost = (values["lossyear"] > 0) & (values["lossyear"] <= spec["year"] - 2000)
        no_data = values["datamask"] == NODATA
        for threshold, label in [(30, spec["label"]), (10, "Hansen GFC (> 10 %)")]:
            forest[label] = ((values["treecover2000"] > threshold) & ~lost).astype(float).mask(no_data)
        continue
    native = values[str(spec["year"])]
    no_data = native.isin([NODATA, *spec["nodata"]])
    forest[spec["label"]] = native.isin(spec["forest_classes"]).astype(float).mask(no_data)
    if "crosswalk" in spec:
        lulc[spec["label"]] = native.map(spec["crosswalk"]).where(~no_data)
forest = pd.DataFrame(forest)
lulc = pd.DataFrame(lulc)
print(f"{len(points):,} points representing {TOTAL_HA:,.0f} ha")

## 9. How the sample works and how uncertain the areas are

### 9.1 Design

- Guinea is covered by **H3 resolution-7 hexagons** (~434 ha each). The **centre of every hexagon** whose centre is
  in Guinea is a sample point: **56,712 points**, one every ~2.2 km, in a regular (systematic) pattern.
- Each product is read at the 10/30/100 m pixel under each point.
- Each point stands for its hexagon: its weight $w_i$ is the hexagon's area.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [1.3, 1]})
for name in ECOREGION_NAMES:
    chosen = points[points["ECO_NAME"] == name]
    axes[0].scatter(chosen["lon"], chosen["lat"], s=0.3, color=ECOREGION_COLORS_MPL[name],
                    label=f"{name}: {len(chosen):,} points")
map_frame(axes[0], f"{len(points):,} sample points (centres of H3 resolution-7 hexagons)")
axes[0].legend(markerscale=20, fontsize=8, loc="lower left")

DETAIL = (-10.95, 10.05, -10.65, 10.30)  # a small window to show the design
in_window = points[points["lon"].between(DETAIL[0], DETAIL[2]) & points["lat"].between(DETAIL[1], DETAIL[3])]
for cell in in_window["h3_id"]:
    outline = np.array([(lng, lat) for lat, lng in h3.cell_to_boundary(cell)] + [h3.cell_to_boundary(cell)[0][::-1]])
    axes[1].plot(outline[:, 0], outline[:, 1], color="#999999", linewidth=0.6)
axes[1].scatter(in_window["lon"], in_window["lat"], s=18, color="#c0392b", zorder=3, label="sample point (hexagon centre)")
map_frame(axes[1], "Detail: each point represents its hexagon (~434 ha)", DETAIL)
axes[1].legend(loc="lower left", fontsize=9)
axes[0].add_patch(plt.Rectangle(DETAIL[:2], DETAIL[2] - DETAIL[0], DETAIL[3] - DETAIL[1], fill=False, edgecolor="blue", linewidth=1.5))
plt.tight_layout()
plt.show()

points.groupby("ECO_NAME").agg(points=("h3_id", "size"), area_ha=("weight_km2", lambda w: (w * HA_PER_KM2).sum())).round(0)

### 9.2 Area and its uncertainty

For a class $c$ (e.g. forest in a product), with $y_i = 1$ if point $i$ is in the class and 0 otherwise:

$$\hat{p}_c = \frac{\sum_i w_i\, y_i}{\sum_i w_i}, \qquad \hat{A}_c = A \,\hat{p}_c, \qquad
SE(\hat{A}_c) = A \sqrt{\frac{\hat{p}_c (1 - \hat{p}_c)}{n}}, \qquad \text{95 \% CI} = \hat{A}_c \pm 1.96\, SE$$

where $A$ is Guinea's area and $n$ the number of points. This is the sample-based area estimator recommended by
Olofsson et al. (2014) and used for Guinea's REDD+ activity data (NERF 2026). The systematic sample is treated as a
simple random one, which slightly overstates the error (Stehman 2009). It estimates **the area a map gives**, not
the true area: the map's own errors are a separate question (reference labels, `lulc_benchmark.ipynb` section 10).

The uncertainty depends on how common the class is: the relative error grows quickly for rare classes.

In [ ]:
def area_estimate(indicator, point_weights=weights_ha):
    """Area (ha) where `indicator` is 1, and its standard error. Missing values count as 0."""
    indicator = indicator.fillna(0).astype(float)
    point_weights = point_weights.reindex(indicator.index)
    share = np.average(indicator, weights=point_weights)
    total = point_weights.sum()
    return total * share, total * np.sqrt(share * (1 - share) / len(indicator))


# Worked example: built-up area in Esri 2020
built_up = (lulc["Esri LULC"] == 4).astype(float)
area, se = area_estimate(built_up)
print(f"Esri built-up 2020: {int(built_up.sum())} of {len(built_up):,} points → p = {area / TOTAL_HA:.4f}")
print(f"  area = {TOTAL_HA:,.0f} ha x {area / TOTAL_HA:.4f} = {area:,.0f} ha, SE = {se:,.0f} ha, "
      f"95 % CI = {area - 1.96 * se:,.0f} – {area + 1.96 * se:,.0f} ha (± {196 * se / area:.0f} %)")

shares = np.logspace(-3.3, np.log10(0.6), 200)
relative_ci = 196 * np.sqrt(shares * (1 - shares) / len(points)) / shares
examples = []
for label in LULC_KEYS:
    name = PRODUCTS[label]["label"]
    for code_, cls in HARMONISED_CLASSES.items():
        a, s = area_estimate((lulc[name] == code_).astype(float))
        if a > 0:
            examples.append({"product": name, "class": cls, "share_%": 100 * a / TOTAL_HA, "ci_%": 196 * s / a, "area_ha": a})
examples = pd.DataFrame(examples)
fig = go.Figure(go.Scatter(x=100 * shares, y=relative_ci, mode="lines", line={"color": "grey"},
                           name=f"n = {len(points):,} points", hovertemplate="%{x:.2f} % of Guinea → ± %{y:.1f} %<extra></extra>"))
for cls, group in examples.groupby("class"):
    fig.add_trace(go.Scatter(x=group["share_%"], y=group["ci_%"], mode="markers", name=cls, customdata=group[["product", "area_ha"]],
                             marker={"color": HARMONISED_COLORS[cls], "size": 10},
                             hovertemplate="%{customdata[0]}: %{customdata[1]:,.0f} ha, ± %{y:.1f} %<extra></extra>"))
fig.update_xaxes(type="log", title="Class area (% of Guinea)")
fig.update_yaxes(type="log", title="95 % confidence interval (± % of the area)")
fig.update_layout(title="Uncertainty of a class area with 56,712 points (classes of Esri, Dynamic World, WorldCover 2020)",
                  template=PLOT_TEMPLATE, height=500)
fig.show()

### 9.3 Check against a wall-to-wall count

Hansen was counted on all ~320 million pixels (Part I). The same quantities estimated from the 56,712 points should
fall inside their 95 % interval about 95 % of the time.

In [ ]:
hansen_points = pd.read_parquet(BENCHMARK_DIR / "hansen_h3res7.parquet").set_index("point_id").reindex(points.index)
forest_2000_points = (hansen_points["treecover2000"] > CANOPY_THRESHOLD).astype(float)
rows = [{"quantity": "Forest 2000", "wall_to_wall_ha": forest_2000_ha,
         **dict(zip(["sample_ha", "se_ha"], area_estimate(forest_2000_points)))}]
for year in range(METHOD_CHANGE_YEAR, LAST_YEAR + 1):
    lost = ((hansen_points["treecover2000"] > CANOPY_THRESHOLD) & (hansen_points["lossyear"] == year - 2000)).astype(float)
    rows.append({"quantity": f"Loss {year}", "wall_to_wall_ha": national_loss[year],
                 **dict(zip(["sample_ha", "se_ha"], area_estimate(lost)))})
sample_check = pd.DataFrame(rows)
sample_check["inside_95%_CI"] = (sample_check["sample_ha"] - sample_check["wall_to_wall_ha"]).abs() <= 1.96 * sample_check["se_ha"]
print(f"{sample_check['inside_95%_CI'].mean():.0%} of the estimates contain the wall-to-wall value (expected ~95 %)")

loss_check = sample_check[sample_check["quantity"].str.startswith("Loss")]
fig = go.Figure()
fig.add_trace(go.Scatter(x=loss_check["quantity"].str[-4:].astype(int), y=loss_check["sample_ha"],
                         error_y={"array": 1.96 * loss_check["se_ha"]}, mode="markers", marker={"size": 9, "color": "#c0392b"},
                         name="Sample estimate (95 % CI)", hovertemplate="%{x}: %{y:,.0f} ha<extra></extra>"))
fig.add_trace(go.Scatter(x=loss_check["quantity"].str[-4:].astype(int), y=loss_check["wall_to_wall_ha"], mode="lines+markers",
                         line={"color": "black"}, name="Wall-to-wall count", hovertemplate="%{x}: %{y:,.0f} ha<extra></extra>"))
fig.update_layout(title="Hansen annual loss: 56,712 points vs every pixel", xaxis_title="Year", yaxis_title="Loss (ha)",
                  yaxis_tickformat=",.0f", template=PLOT_TEMPLATE)
fig.show()
sample_check.round(0)

## 10. How much the land-cover products differ: built-up, grassland and cropland

Area of each class in 2020 according to the three 10 m products, with 95 % confidence intervals. When the intervals
do not overlap, the difference is a real difference between the maps, not sampling noise.

In [ ]:
FOCUS_CLASSES = {4: "Built-up", 2: "Shrub & grassland", 3: "Cropland"}
rows = []
for name in lulc.columns:
    for code_, cls in HARMONISED_CLASSES.items():
        a, s = area_estimate((lulc[name] == code_).astype(float))
        rows.append({"product": name, "class": cls, "area_ha": a, "ci95_ha": 1.96 * s})
class_areas = pd.DataFrame(rows)

fig = make_subplots(rows=1, cols=3, subplot_titles=list(FOCUS_CLASSES.values()))
for column, cls in enumerate(FOCUS_CLASSES.values(), start=1):
    part = class_areas[class_areas["class"] == cls]
    fig.add_trace(go.Bar(x=part["product"], y=part["area_ha"], error_y={"array": part["ci95_ha"]},
                         marker_color=[PRODUCT_COLORS[p] for p in part["product"]], showlegend=False,
                         hovertemplate="%{x}: %{y:,.0f} ha ± %{error_y.array:,.0f}<extra></extra>"), row=1, col=column)
fig.update_yaxes(title_text="Area in 2020 (ha)", tickformat=",.0f", col=1)
fig.update_yaxes(tickformat=",.0f")
fig.update_layout(title="Built-up, grassland and cropland in the three 10 m land-cover products (95 % CI)",
                  template=PLOT_TEMPLATE, height=480)
fig.show()

fig = px.bar(class_areas, y="product", x="area_ha", color="class", orientation="h", color_discrete_map=HARMONISED_COLORS,
             category_orders={"class": list(HARMONISED_CLASSES.values())},
             labels={"area_ha": "Area (ha)", "product": "", "class": ""},
             title="Land cover of Guinea in 2020 according to each product", template=PLOT_TEMPLATE, height=350)
fig.update_xaxes(tickformat=",.0f")
fig.update_traces(hovertemplate="%{x:,.0f} ha")
fig.show()
class_areas.pivot(index="class", columns="product", values="area_ha").loc[list(HARMONISED_CLASSES.values())].round(0)

### 10.1 Do they put the class in the same places?

Equal areas could still be in different places. For each class, **overlap** = area all three products give to the class
/ area at least one gives to it (Jaccard index). 100 % = the same map, 0 % = no place in common. The pairwise columns
compare two products at a time.

In [ ]:
rows = []
valid = lulc.notna().all(axis=1)
w = weights_ha[valid]
for code_, cls in HARMONISED_CLASSES.items():
    in_class = lulc[valid] == code_
    row = {"class": cls,
           "any_product_ha": w[in_class.any(axis=1)].sum(),
           "all_three_ha": w[in_class.all(axis=1)].sum()}
    row["overlap_all_three_%"] = 100 * row["all_three_ha"] / row["any_product_ha"] if row["any_product_ha"] else np.nan
    for a, b in [("Esri LULC", "Dynamic World"), ("Esri LULC", "ESA WorldCover"), ("Dynamic World", "ESA WorldCover")]:
        union = w[in_class[a] | in_class[b]].sum()
        row[f"{a} / {b} %"] = 100 * w[in_class[a] & in_class[b]].sum() / union if union else np.nan
    rows.append(row)
class_overlap = pd.DataFrame(rows).set_index("class")

fig = px.bar(class_overlap.reset_index(), x="class", y="overlap_all_three_%", color="class",
             color_discrete_map=HARMONISED_COLORS, custom_data=["all_three_ha", "any_product_ha"],
             labels={"overlap_all_three_%": "Overlap of the three products (%)", "class": ""},
             title="Where Esri, Dynamic World and WorldCover agree on a class (2020)", template=PLOT_TEMPLATE)
fig.update_traces(hovertemplate="%{y:.0f} %: %{customdata[0]:,.0f} ha in all three of %{customdata[1]:,.0f} ha in any")
fig.update_layout(showlegend=False, yaxis_range=[0, 100])
fig.show()
class_overlap.round(1)

## 11. Forest area in 2020: all products

The three land-cover products plus four forest products. Hansen is shown with the adopted 30 % threshold and with the
legal 10 % (Code forestier 2017). The dashed line is the official forest area reported to FAO (FRA 2020: 6.19
million ha in 2020, extrapolated from a 1988 vegetation map).

In [ ]:
FRA_2020_FOREST_HA = 6_189_000
rows = []
for name in forest.columns:
    a, s = area_estimate(forest[name])
    group = "10 m land cover" if name in lulc.columns else "forest product"
    rows.append({"product": name, "group": group, "forest_ha": a, "ci95_ha": 1.96 * s,
                 "forest_%_of_guinea": 100 * a / TOTAL_HA, "no_data_%": 100 * forest[name].isna().mean()})
forest_areas = pd.DataFrame(rows).sort_values("forest_ha")

fig = go.Figure(go.Bar(y=forest_areas["product"], x=forest_areas["forest_ha"], orientation="h",
                       error_x={"array": forest_areas["ci95_ha"]}, marker_color=[PRODUCT_COLORS[p] for p in forest_areas["product"]],
                       customdata=forest_areas[["forest_%_of_guinea", "group"]],
                       hovertemplate="%{y} (%{customdata[1]}): %{x:,.0f} ha, %{customdata[0]:.1f} % of Guinea<extra></extra>"))
fig.add_vline(x=FRA_2020_FOREST_HA, line_dash="dash", line_color="black")
fig.add_annotation(x=FRA_2020_FOREST_HA, y=1.02, yref="paper", text="FAO FRA 2020 (official)", showarrow=False)
fig.update_layout(title="Forest area of Guinea around 2020 (95 % CI)", xaxis_title="Forest (ha)", xaxis_tickformat=",.0f",
                  template=PLOT_TEMPLATE, height=480)
fig.show()
forest_areas.set_index("product").round(1)

## 12. Forest area per ecoregion in 2020

In [ ]:
forest_ecoregion = (forest.fillna(0).mul(weights_ha, axis=0).groupby(points["ECO_NAME"]).sum()
                    .drop(index="(outside the ecoregion polygons)", errors="ignore"))
plot_table = forest_ecoregion.reset_index(names="ECO_NAME").melt(id_vars="ECO_NAME", var_name="product", value_name="forest_ha")
fig = px.bar(plot_table, x="ECO_NAME", y="forest_ha", color="product", barmode="group",
             category_orders={"product": list(forest_areas.sort_values("forest_ha", ascending=False)["product"])},
             color_discrete_map=PRODUCT_COLORS,
             labels={"forest_ha": "Forest in 2020 (ha)", "ECO_NAME": "", "product": "Product"},
             title="Forest area per ecoregion in 2020 according to each product", template=PLOT_TEMPLATE, height=550)
fig.update_yaxes(tickformat=",.0f")
fig.update_traces(hovertemplate="%{y:,.0f} ha")
fig.show()

ecoregion_area = weights_ha.groupby(points["ECO_NAME"]).sum().reindex(forest_ecoregion.index)
display(forest_ecoregion.round(0))
print("Forest as % of the ecoregion:")
(100 * forest_ecoregion.div(ecoregion_area, axis=0)).round(1)

## 13. Where the forest maps disagree

Following `lulc_benchmark.ipynb` (section 8.5), for the seven products of this report (Hansen at 30 %): a point is
*agreed forest* if all products with data call it forest, *agreed non-forest* if none does, and *disagreement*
otherwise. The maps show, per resolution-5 hexagon (~25,000 ha), the share of the area where the products disagree
and the average share of products calling it forest.

In [ ]:
REPORT_FOREST = [PRODUCTS[key]["label"] for key in FOREST_KEYS]
votes = forest[REPORT_FOREST]
forest_votes = (votes == 1).sum(axis=1)
valid_votes = votes.notna().sum(axis=1)
stratum = pd.Series(np.select([valid_votes == 0, forest_votes == valid_votes, forest_votes == 0],
                              ["no data", "agreed forest", "agreed non-forest"], "disagreement"), index=votes.index)

hexagon5 = points["h3_res5_id"]
hex_weight = weights_ha.groupby(hexagon5).sum()
disagreement = pd.DataFrame({
    "disagreement_%": 100 * weights_ha.where(stratum == "disagreement", 0).groupby(hexagon5).sum() / hex_weight,
    "products_saying_forest_%": 100 * (weights_ha * forest_votes / valid_votes.replace(0, np.nan)).groupby(hexagon5).sum() / hex_weight,
})
disagreement = gpd.GeoDataFrame(disagreement, geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(cell)])
                                                        for cell in disagreement.index], crs="EPSG:4326")

fig, axes = plt.subplots(1, 2, figsize=(19, 7))
disagreement.plot(column="disagreement_%", cmap="Purples", vmin=0, vmax=100, ax=axes[0], linewidth=0,
                  legend=True, legend_kwds={"label": "Products disagree (% of hexagon)", "shrink": 0.7})
map_frame(axes[0], f"Where the forest maps disagree ({COMPARISON_YEAR}), H3 resolution 5")
disagreement.plot(column="products_saying_forest_%", cmap="Greens", vmin=0, vmax=100, ax=axes[1], linewidth=0,
                  legend=True, legend_kwds={"label": "Products calling it forest (%)", "shrink": 0.7})
map_frame(axes[1], f"Average share of the {len(REPORT_FOREST)} products calling the area forest")
plt.tight_layout()
plt.show()

by_stratum = (pd.crosstab(points["ECO_NAME"], stratum, values=weights_ha, aggfunc="sum").fillna(0)
              .drop(index="(outside the ecoregion polygons)", errors="ignore"))
by_stratum.loc["Guinea"] = by_stratum.sum()
by_stratum_pct = 100 * by_stratum.div(by_stratum.sum(axis=1), axis=0)
plot_table = by_stratum_pct.reset_index(names="ECO_NAME").melt(id_vars="ECO_NAME", var_name="stratum", value_name="share")
fig = px.bar(plot_table, y="ECO_NAME", x="share", color="stratum", orientation="h",
             color_discrete_map={"agreed forest": "#2e7d32", "agreed non-forest": "#c5b358", "disagreement": "#7b4ea3", "no data": "#cccccc"},
             category_orders={"stratum": ["agreed forest", "disagreement", "agreed non-forest", "no data"]},
             labels={"share": "% of the area", "ECO_NAME": "", "stratum": ""},
             title=f"Agreement of the {len(REPORT_FOREST)} forest maps per ecoregion ({COMPARISON_YEAR})", template=PLOT_TEMPLATE)
fig.update_traces(hovertemplate="%{x:.1f} %")
fig.show()
by_stratum.round(0)

## Findings

Numbers from the run of 21 October 2026 (Hansen v1.13, canopy > 30 % unless stated).

**Part I – Hansen**

1. **The forest definition changes the answer by a factor of ~3–10.** Forest in 2000 is 22.5 million ha at canopy
   > 10 % (93 % of Guinea), 8.17 million ha at > 30 % (34 %) and 2.17 million ha at > 50 % (9 %). The effect is
   largest in the West Sudanian savanna: 5.06 million ha of "forest" at > 10 %, 0.26 million ha at > 30 %.
   Loss 2013–2025 is 3.42 million ha at > 10 %, 2.16 million ha at > 30 % and 0.63 million ha at > 50 %.
2. **The 2013 method change is an eight-fold jump.** Mean loss was 21,130 ha/year in 2001–2012 and 165,850 ha/year in
   2013–2025 (147,570 ha/year in 2020–2025). Every release from 2015 to 2025 reports the same values for past years,
   so the break cannot be fixed by using another release. Only 2013 onwards is used.
3. **Forest in 2025: 5.76 million ha** (23.7 % of Guinea), 63 % of it in the Guinean forest-savanna (3.61 million ha)
   and 20 % in the Western Guinean lowland forests (1.15 million ha, the densest: 53 % cover).
4. **All forested ecoregions lose forest at about the same rate**: 2.4–2.5 % of their remaining forest per year in
   2013–2025 (Guinea 2.4 %). The lowland forests have the highest recent rate (2.55 %/year in 2020–2025). The West
   Sudanian savanna is lower but rising (1.2 → 1.7 %/year).
5. **Loss is strongly concentrated in space** (H3 resolution 8, ~62 ha): 41 % of Guinea has no loss at all. Half of
   the 2013–2025 loss falls in 8.6 % of the land, 80 % in 20 % of the land. Against the **forest** area, 40 % of the
   2013 forest holds 74 % of the loss (79 % for 2020–2025) and 50 % holds 84 %. Coarser zones hide part of this
   (resolution 6: 76 %).
6. **The hotspots are in the west and the south-east.** The largest cluster is in the western forest-savanna and
   montane forests around Télimélé, north of Kindia and west of Mamou (~10.3–11.2° N, 12.2–13.5° W); the densest
   single zone is along the Sierra Leone border east of Forécariah (~9.4° N, 12.9° W); the third is Guinée
   forestière in the south-east. By ecoregion, the core hotspots (50 % of the loss) are 62 % in the forest-savanna,
   24 % in the lowland forests and 12 % in the montane forests.

**Part II – Land-cover benchmark**

7. **The sample works.** 56,712 points reproduce Hansen's full pixel count for forest 2000 and for every year of
   loss 2013–2025 (14 of 14 within the 95 % interval). A common class is estimated to ± 1–2 %; a rare one like
   built-up (1 % of Guinea) to ± 8 %.
8. **The three 10 m products disagree on the minor classes far more than the sampling error.** Built-up: 91,000 ha
   (WorldCover) to 266,000 ha (Esri), × 2.9. Cropland: 248,000 ha (Esri) to 537,000 ha (WorldCover), × 2.2.
   Shrub & grassland: 8.5 (Dynamic World) to 13.2 million ha (WorldCover). They also put the classes in different
   places: all three agree on only 4.7 % of the area any of them calls cropland, and 23 % for built-up.
9. **Forest around 2020 ranges from 6.4 to 22.0 million ha** depending on the product: Hansen > 30 % (6.4, the closest
   to the official FAO FRA 6.19), WorldCover 10.4, JRC GFC2020 12.5, Esri 13.9, Dynamic World 15.2, Copernicus 18.2,
   Hansen > 10 % 20.0, GLC_FCS30D 22.0. The spread comes mostly from the forest definition (dense vs open
   woodland), not from map errors.
10. **The seven forest maps agree on only 20 % of Guinea** (13.5 % agreed forest, 6 % agreed non-forest); they
    disagree on 80 %. Agreement is best in the closed forest block around 9.5–10.5° N, 9.5–10.5° W and in parts of the
    south-east; the northern savanna and the forest-savanna mosaic are where the choice of definition matters most.

## References

- FAO (2020). *Évaluation des ressources forestières mondiales 2020 – Rapport Guinée*. Rome.
- Hansen, M. C., et al. (2013). High-resolution global maps of 21st-century forest cover change. *Science*, 342(6160), 850–853.
- Olofsson, P., Foody, G. M., Herold, M., Stehman, S. V., Woodcock, C. E., & Wulder, M. A. (2014). Good practices for
  estimating area and assessing accuracy of land change. *Remote Sensing of Environment*, 148, 42–57.
- République de Guinée (2017). Loi L/2017/060/AN portant Code forestier.
- République de Guinée (2026). *Niveau de référence des forêts (NRF) et niveau d'émissions de référence des forêts
  (NERF) de la Guinée, période 2016–2020*. Submitted to the UNFCCC.
- Stehman, S. V. (2009). Sampling designs for accuracy assessment of land cover. *International Journal of Remote
  Sensing*, 30(20), 5243–5272.
- Uber Technologies (2018). H3: a hexagonal hierarchical geospatial indexing system. https://h3geo.org